### Query Translation - RAG Fusion
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some
way as to improve retrival.

Semantic search on embeddings is hard to get right. Embedding long documents is
especially challenging. User queries are a challenge too. If the user provides an ambigious
query, they'll end up get an ambiguous matches from embeddings and consequently an ambguous answer. The ambiguous matches land up in the LLM's context from which comes the LLM's response, which could lead to hallucinations. 

There are 2 broad approaches to tackle the above issue:
1. Multi Query
2. RAG Fusion

In this workbook, we develop the **RAG Fusion technique**.

**What is it?**

A **retrieval re-ranking technique** inspired by “Reciprocal Rank Fusion” (RRF) in information retrieval. You run multiple retrievals (often: multiple queries, multiple retrievers, or both), then **fuse the ranked lists** of results into one final ranking.

**How it works:**

* Each retrieval returns a ranked list (doc A rank=1, doc B rank=2, etc).
* Fusion scores docs by combining their **reciprocal ranks**:

$$
score(d) = \sum_{retrievers} \frac{1}{k+rank(d)}
$$
(with `k`= smoothening constant)
* Documents that appear across multiple queries/retrievers rise to the top.
* Reduces noise because only documents consistently relevant get boosted.

**Key Point:** RAG Fusion improves precision and robustness by rewarding cross-query/retriever consensus.

The diagram below illustrates this technique.

![Multi Query](images/rag_fusion.png)

In [2]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [3]:
# load API keys from .env files
load_dotenv(override=True)
# for colorful text output
console = Console()

In [4]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_rf"

In [5]:
def create_or_load_embeddings():
    """creates if not available or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [6]:
retriever = create_or_load_embeddings()

Loading document from URL https://lilianweng.github.io/posts/2023-06-23-agent/. Please wait...

Loaded 1 documents from URL

Metadata of first document: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}

First 200 chars of first document: 

      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a 

Chunking the PDF. Please wait...

Created 50 chunks

Creating embeddings. Please wait...

Local embeddings created at c:\Dev\Code\git-projects\learning_langchain\src\langchain_tutorial\faiss_index_rag_rf

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# RAG-Fusion: prompt
template = """You are a helpful assistant that generates multiple search queries 
based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output ({num_queries} queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [8]:
generate_queries = (
    prompt_rag_fusion | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke(
    {
        "num_queries": 5,
        "question": "What is task decomposition for LLM agents?",
    }
)

['Here are 5 search queries related to "What is task decomposition for LLM agents?":',
 '',
 '1.  **Task decomposition techniques for LLM agents**',
 '2.  **How do LLMs break down complex tasks for autonomous agents?**',
 '3.  **Strategies for sub-task generation in large language model agents**',
 '4.  **Hierarchical planning methods for LLM-based AI agents**',
 '5.  **Benefits of task decomposition in LLM agent performance**']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [9]:
from langchain.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    """Reciprocal_rank_fusion that takes multiple lists of ranked documents
    and an optional parameter k used in the RRF formula"""

    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

In [20]:
# Retrieve
from pydoc import doc


question = "What is task decomposition for LLM agents?"
# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain.invoke({"question": question, "num_queries": 5})
print(f"Got {len(docs)} documents")
# for i in range(5):
#     print(f"{docs[i]}\n\n")

# Note: docs -> List[(document, score)] (i.e. a list of tuples of (Document, score))
for i, (doc, score) in enumerate(docs):
    print(f"Document #{i+1}\nContent: {doc.page_content}\nScore: {score}")

# # and print the retrival chain too
# console.print(f"Retrieval chain: {retrieval_chain}")

Got 12 documents
Document #1
Content: }
]
Challenges#
After going through key ideas and demos of building LLM-centered agents, I start to see a couple common limitations:
Score: 0.09918032786885246
Document #2
Content: Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structur

In [10]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = final_rag_chain.invoke({"question": question})
console.print(Markdown(response))

Task decomposition for LLM agents is the process of breaking down complex tasks into smaller, simpler, and more    
manageable steps or subgoals. This allows the agent to handle complicated tasks more efficiently.                  

Key aspects include:                                                                                               

 • Chain of Thought (CoT): A standard prompting technique where the model is instructed to "think step by step" to 
   decompose hard tasks.                                                                                           
 • Tree of Thoughts: An extension of CoT that explores multiple reasoning possibilities at each step, creating a   
   tree structure of thoughts.                                                                                     
 • Methods: Task decomposition can be achieved by the LLM itself with simple prompts (e.g., "Steps for XYZ"), using
   task-specific instructions, or with human input.